# Fed Chair speeches/testimony -- postprocess (the clean, tidy corpus)

Takes the raw compiled dataset (one row per PDF FILE -- a small number of
items are split across multiple PDF parts) and produces the final
analysis-ready dataset: one row per DOCUMENT. Steps: merge multi-part items
into a single text (ordered by part number), clean whitespace, derive final
metadata, drop exact-duplicate texts, flag genuinely unreadable
(scanned-image) documents rather than dropping them.

In [1]:
import os, sys
import pandas as pd
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), "Helpers"))
import fed_speeches_helpers as fsh

BASE = os.path.dirname(os.getcwd())
COMPILED_PATH = os.path.join(BASE, "Data Input", "Raw Input", "fed_speeches_compiled_raw.csv")
CLEAN_PATH = os.path.join(BASE, "Data Output", "fed_speeches_clean.csv")

## Load the compiled raw dataset

In [2]:
df = pd.read_csv(COMPILED_PATH)
print(df.shape)
print(f"{df['item_id'].nunique():,} distinct items ({len(df) - df['item_id'].nunique()} multi-part items)")

(2087, 13)
2,082 distinct items (5 multi-part items)


## Merge multi-part items, clean text, dedupe, finalize schema

In [3]:
tidy = fsh.build_speeches_clean(df)
print(tidy.shape)
print(f"needs_ocr: {tidy['needs_ocr'].sum()}")
print(tidy['doc_type'].value_counts())
print(f"word_count: min={tidy['word_count'].min()}, max={tidy['word_count'].max()}, median={tidy['word_count'].median()}")

(2082, 14)
needs_ocr: 6
doc_type
speech                  1946
testimony                106
transcript                15
lecture                    9
educational resource       3
press_conference           3
Name: count, dtype: int64
word_count: min=0, max=74576, median=2885.5


## Save

In [4]:
os.makedirs(os.path.dirname(CLEAN_PATH), exist_ok=True)
tidy.to_csv(CLEAN_PATH, index=False)
print(f"Saved {len(tidy):,} rows to: {CLEAN_PATH}")

Saved 2,082 rows to: C:\Users\User\Desktop\DISSERTATION PAPERS\FED_TREASURY_DATA\Data Output\fed_speeches_clean.csv
